#
# Universidad EAFIT
# 2026-2
# SI7016 - NLP Aplicado - Lecture 05c - Agentes
#

# Ejemplo 1: Creación de un Agente LLM en Python

In [ ]:
# instalar dependencias
%pip install -U langchain langchain-openai langchain-community langgraph wikipedia openai faiss-cpu tiktoken

In [ ]:
# api keys (desde variables de entorno o solicitadas de forma interactiva)
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# Necesario más abajo, en el Ejemplo 4 (LlamaIndex + Hugging Face Inference Providers)
HF_TOKEN = os.environ.get("HF_TOKEN")

# Opcional: solo si quieres ver las trazas de este notebook en LangSmith
# (https://smith.langchain.com) - no es necesario para que los agentes funcionen.
# if "LANGCHAIN_API_KEY" not in os.environ:
#     os.environ["LANGCHAIN_API_KEY"] = getpass("LangSmith API Key (opcional): ")
#     os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Nota 2026: la celda original pedía también LANGCHAIN_API_KEY y SERPAPI_API_KEY
# con os.environ["..."] a secas - eso lanza un KeyError inmediato si esas
# variables no existen, aunque el Ejemplo 1 ni siquiera las usa. Cada API key
# se solicita ahora justo donde se necesita (SERPAPI_API_KEY está más abajo,
# en el Ejemplo 2).

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.checkpoint.memory import InMemorySaver
import json

# --- Herramienta: Wikipedia (en español) ---
wiki = WikipediaAPIWrapper(lang="es", top_k_results=3, doc_content_chars_max=4000)


@tool("wikipedia")
def search_wikipedia(query: str) -> str:
    """Busca información en Wikipedia (ES). Devuelve un resumen breve de los resultados."""
    try:
        return wiki.run(query)
    except json.JSONDecodeError as e:
        # This error typically means the response from Wikipedia was not valid JSON.
        # It could be a transient API issue, a network problem, or a malformed query.
        return f"Error al buscar en Wikipedia para '{query}': El servidor devolvió una respuesta no válida (JSONDecodeError: {e}). Por favor, intenta de nuevo o verifica la consulta."
    except Exception as e:
        # Catch other potential errors from the Wikipedia API wrapper
        return f"Error inesperado al buscar en Wikipedia para '{query}': {e}. Por favor, intenta de nuevo."


# --- Agente ReAct con herramientas + memoria ---
# Nota 2026: `create_react_agent` + `AgentExecutor` (langchain.agents), el
# prompt de hub `hwchase17/react-chat` y `RunnableWithMessageHistory` +
# `ChatMessageHistory` están deprecados. `create_agent` es la fábrica
# unificada actual de LangChain v1 (mismo patrón que
# `class03a-langchain-agent.ipynb`): compila un grafo de LangGraph por debajo
# y usa un `checkpointer` para memoria persistente por `thread_id`, sin
# necesidad de un prompt externo ni de un wrapper de historial aparte.
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[search_wikipedia],
    system_prompt="Eres un asistente que responde preguntas usando Wikipedia cuando sea útil.",
    checkpointer=InMemorySaver(),
)

# --- Ejemplo de interacción ---
session_id = "demo-profesor"  # cámbialo por el id de usuario/sesión que quieras persistir
config = {"configurable": {"thread_id": session_id}}

resp1 = agent.invoke(
    {"messages": [{"role": "user", "content": "¿Quién fue Alan Turing? Por favor, resume en 3 líneas y cita la fuente si usas Wikipedia."}]},
    config=config,
)
print(resp1["messages"][-1].content)

# La memoria (vía checkpointer + thread_id) guarda el contexto para la siguiente pregunta:
resp2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Amplía con 2 aportes clave adicionales y una fecha importante."}]},
    config=config,
)
print(resp2["messages"][-1].content)


# Ejemplo 2: Agente con Razonamiento y Planificación

In [ ]:
%pip install google-search-results

# Este agente puede recibir objetivos y descomponerlos en tareas, buscando información actual en la web.

# se requiere la API KEY de SerpAPI - https://serpapi.com

# realice el registro en línea, tiene 100 búsquedas gratuitas al mes, uso no comercial

# la celda de código de abajo ya pide la clave de forma interactiva (getpass) si no
# está definida como variable de entorno - no hace falta exportarla a mano.

In [ ]:
import os
from getpass import getpass
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper
from langgraph.checkpoint.memory import InMemorySaver

if "SERPAPI_API_KEY" not in os.environ:
    os.environ["SERPAPI_API_KEY"] = getpass("SerpAPI API Key: ")

search = SerpAPIWrapper()


@tool("buscar_en_la_web")
def buscar_en_la_web(query: str) -> str:
    """Busca en la web información actual (noticias, eventos recientes, etc.) usando SerpAPI."""
    return search.run(query)


# Nota 2026: `initialize_agent` + `AgentType.ZERO_SHOT_REACT_DESCRIPTION`
# (langchain.agents) están deprecados desde LangChain 0.1 y ya no existen en
# LangChain v1 - `load_tools(["serpapi"], ...)` se reemplaza por instanciar la
# herramienta directamente (arriba). `create_agent` reemplaza también al
# system_message aparte: se pasa directo como `system_prompt`.
agent_executor = create_agent(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    tools=[buscar_en_la_web],
    system_prompt="Eres un agente de IA que descompone problemas complejos en pasos más pequeños y ejecuta acciones.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "demo-planificacion"}}
result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Encuentra las últimas noticias sobre inteligencia artificial."}]},
    config=config,
)
print(result["messages"][-1].content)

# Ejemplo 3: OpenAI Agents SDK (multiagentes, tool calling y handoffs)

El [Agents SDK de OpenAI](https://openai.github.io/openai-agents-python/) (paquete `openai-agents`, se importa como `agents`) es un framework ligero, agnóstico de proveedor, centrado en tres primitivas: `Agent` (instrucciones + modelo + herramientas), `Runner` (ejecuta el agente hasta obtener una respuesta final) y `handoff` (delega la conversación a otro agente especializado).

Los ejemplos completos y ejecutables como script están en la carpeta `hf-agent-course/` de este mismo repo:

- `helloworld.py` - agente mínimo, ejecución síncrona (`Runner.run_sync`).
- `weather.py` - un agente con una `@function_tool`.
- `handoffs.py` - un agente de triage que delega a un agente en español o en inglés según el idioma.
- `customerservice.py` / `main.py` - ejemplo más completo (aerolínea): contexto tipado (`RunContextWrapper`), varias herramientas, handoffs con hook (`on_handoff`) y `trace()` para agrupar una conversación completa en una traza.

Aquí en el notebook solo mostramos el patrón de ejecución **asíncrona**, porque es el que cambia según el entorno (notebook vs. script).

In [ ]:
%pip install openai-agents

In [ ]:
# Patrón asíncrono dentro de un notebook (Jupyter/Colab ya soportan `await`
# a nivel de celda - no hace falta nest_asyncio ni asyncio.run aquí).
from agents import Agent, Runner

agent = Agent(name="Assistant", instructions="You are a helpful assistant", model="gpt-5.6")

result = await Runner.run(agent, "Write a haiku about recursion in programming.")
print(result.final_output)

Como **script aparte** (fuera de un notebook) no existe un event loop corriendo de antemano, así que hay que crear uno con `asyncio.run(...)` - es exactamente el patrón de `agent-async.py`:

```python
import asyncio
from agents import Agent, Runner

agent = Agent(name="Assistant", instructions="You are a helpful assistant", model="gpt-5.6")

async def main():
    result = await Runner.run(agent, "Write a haiku about recursion in programming.")
    print(result.final_output)

asyncio.run(main())
```

**Nota:** en versiones antiguas de Jupyter/Colab que no soportaban `await` de nivel superior, había que instalar `nest_asyncio` y llamar `nest_asyncio.apply()` antes de poder usar `asyncio.run()` dentro del notebook. Los notebooks actuales (Jupyter ≥ 7, Colab) ya soportan `await` de forma nativa, así que rara vez hace falta - solo si tu entorno específico da un error de tipo `RuntimeError: this event loop is already running`.

# Ejemplo 4: LlamaIndex + Hugging Face Inference Providers

In [ ]:
%pip install -U llama-index-llms-huggingface-api

In [ ]:
!pip install -U llama-index llama-index-llms-huggingface-api

In [ ]:
# Usar un modelo de código open-weight vía Hugging Face Inference Providers como LLM de chat

import os
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.llms import ChatMessage, MessageRole

HF_TOKEN = os.environ["HF_TOKEN"]  # asegúrate de exportarlo (o defínelo en la celda de arriba)

llm = HuggingFaceInferenceAPI(
    # Modelo de código 2026; si el proveedor no tiene capacidad/lo retira del
    # catálogo, usa una alternativa más liviana como "Qwen/Qwen3.5-9B" o el
    # modelo general "meta-llama/Llama-3.3-70B-Instruct" (usado en
    # class03a-huggingface-agent.ipynb).
    model_name="Qwen/Qwen3-Coder-Next",
    temperature=0.7,
    max_tokens=100,   # alias a max_new_tokens en la API
    token=HF_TOKEN,
    provider="auto",  # deja que Hugging Face elija el proveedor de inferencia disponible
)

# Usa .chat con una lista de mensajes
msgs = [
    ChatMessage(role=MessageRole.SYSTEM, content="You are a helpful coding assistant."),
    ChatMessage(role=MessageRole.USER, content="Hello, how are you?")
]
resp = llm.chat(msgs)

print(resp.message.content)


## Notas / Cierre

- Los tres frameworks de este notebook (LangChain/LangGraph, OpenAI Agents SDK, LlamaIndex) representan tres estilos distintos de construir agentes: grafo explícito con memoria vía checkpointer, primitivas mínimas (Agent/Runner/handoff) centradas en multiagentes, y agentes orientados a RAG/herramientas sobre un LLM servido vía Inference Providers.
- Para **multiagentes con handoffs** (delegar la conversación completa a otro agente especializado, no solo llamar una tool) el patrón más directo hoy es el Agents SDK de OpenAI - ver `hf-agent-course/handoffs.py` y `customerservice.py`. En LangGraph el patrón equivalente es Supervisor vs. Swarm (ver Lecture 05c - slides).
- Para **interoperabilidad entre agentes de distintos frameworks/proveedores** existen ya protocolos abiertos como MCP (herramientas/contexto) y A2A (agente-a-agente) - ver la slide "Protocolos: MCP y A2A" de esta misma lección.
- Los notebooks `components.ipynb` y `tools.ipynb` (carpeta `hf-agent-course/`) profundizan en LlamaIndex: `QueryEngine`, `IngestionPipeline`, `FunctionTool`/`QueryEngineTool`/`ToolSpec` y evaluación/observabilidad con RAGAS-style evaluators y Arize Phoenix.